# CReST: Final Architecture Lock

Validates and locks the final architecture using training and validation systems only, before confirmatory seven-fold training.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Final CReST architecture pilot environment

from pathlib import Path
import json
import random

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import (
    Dataset,
    DataLoader,
    WeightedRandomSampler
)

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score
)

import matplotlib.pyplot as plt


# Paths

BASE_DIR = Path.cwd()

HANDOFF_DIR = (
    BASE_DIR
    / "processed"
    / "crest_v5_handoff"
)

CALIBRATION_DIR = (
    BASE_DIR
    / "processed"
    / "crest_design_calibration"
)

CHECKPOINT_DIR = (
    BASE_DIR
    / "checkpoints"
    / "fold_0"
)

RESULTS_DIR = (
    BASE_DIR
    / "results"
)

FIG_DIR = (
    BASE_DIR
    / "figures"
)


CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

RESULTS_DIR.mkdir(
    exist_ok=True
)

FIG_DIR.mkdir(
    exist_ok=True
)


# Configuration

with open(
    HANDOFF_DIR / "handoff_config.json",
    "r",
    encoding="utf-8"
) as f:

    CONFIG = json.load(f)


GLOBAL_SEED = int(
    CONFIG["global_seed"]
)


def set_global_seed(
    seed=GLOBAL_SEED
):

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_global_seed()


device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


print(
    "Working directory:",
    BASE_DIR
)

print(
    "Device:",
    device
)

print(
    "Environment initialised."
)

Working directory: PROJECT_ROOT
Device: cpu
Final CReST architecture pilot notebook ready.


In [ ]:
# Load modelling data and locked design-calibration artefacts

# Core arrays

credit_tokens = np.load(
    HANDOFF_DIR / "credit_tokens.npy",
    mmap_mode="r"
)

behaviour_tokens = np.load(
    HANDOFF_DIR / "behaviour_tokens.npy",
    mmap_mode="r"
)

performance_mask = np.load(
    HANDOFF_DIR / "performance_mask.npy",
    mmap_mode="r"
)

interaction_mask = np.load(
    HANDOFF_DIR / "interaction_mask.npy",
    mmap_mode="r"
)

student_design_mask = np.load(
    HANDOFF_DIR / "student_design_mask.npy",
    mmap_mode="r"
)

process_tensor = np.load(
    HANDOFF_DIR / "process_tensor.npy",
    mmap_mode="r"
)

booklet_nuisance_ids = np.load(
    HANDOFF_DIR / "booklet_nuisance_ids.npy",
    mmap_mode="r"
)

item_domain_ids = np.load(
    HANDOFF_DIR / "item_domain_ids.npy",
    mmap_mode="r"
)

item_ideation_ids = np.load(
    HANDOFF_DIR / "item_ideation_ids.npy",
    mmap_mode="r"
)


model_context = pd.read_pickle(
    HANDOFF_DIR / "model_context.pkl"
)

primary_meta = pd.read_pickle(
    HANDOFF_DIR / "primary_meta.pkl"
)

context_registry_locked = pd.read_csv(
    HANDOFF_DIR / "context_registry_locked.csv"
)


# Locked Fold-0 calibration artefacts

credit_item_mean = np.load(
    CALIBRATION_DIR
    / "fold0_credit_item_mean.npy"
)

behaviour_item_rates = np.load(
    CALIBRATION_DIR
    / "fold0_behaviour_item_rates.npy"
)

process_item_mean = np.load(
    CALIBRATION_DIR
    / "fold0_process_item_mean.npy"
)

process_item_std = np.load(
    CALIBRATION_DIR
    / "fold0_process_item_std.npy"
)

signature_profiles = np.load(
    CALIBRATION_DIR
    / "fold0_signature_profiles.npy"
)

BOOKLET_TO_SIGNATURE = np.load(
    CALIBRATION_DIR
    / "booklet_to_signature.npy"
)


# Fold 0

fold_values = (
    primary_meta[
        "country_fold"
    ].to_numpy()
)


train_idx = np.where(
    ~np.isin(
        fold_values,
        [0, 1]
    )
)[0]


validation_idx = np.where(
    fold_values == 1
)[0]


test_idx = np.where(
    fold_values == 0
)[0]


# Audit

assert credit_item_mean.shape == (32,)
assert behaviour_item_rates.shape == (32, 3)
assert process_item_mean.shape == (32, 3)
assert process_item_std.shape == (32, 3)
assert signature_profiles.shape == (10, 21)
assert BOOKLET_TO_SIGNATURE.shape == (30,)


print(
    "Train:",
    len(train_idx)
)

print(
    "Validation:",
    len(validation_idx)
)

print(
    "Test untouched:",
    len(test_idx)
)

print(
    "Signature profiles:",
    signature_profiles.shape
)

print(
    "\nLocked calibration artefacts loaded successfully."
)

Train: 99966
Validation: 20949
Test untouched: 21649
Signature profiles: (10, 21)

Locked calibration artefacts loaded successfully.


In [ ]:
# Convert locked 21-D calibration profiles to 3 x 7 process-token profiles

process_ids_np = np.asarray(
    item_ideation_ids,
    dtype=np.int64
)

domain_ids_np = np.asarray(
    item_domain_ids,
    dtype=np.int64
)


# Original 21-D order from Notebook 03:
#
# 0:3   -> credit residuals      [GDI, GCI, EII]
# 3:12  -> behaviour residuals   [3 per process]
# 12:21 -> process residuals     [3 per process]

signature_credit = (
    signature_profiles[:, 0:3]
    .reshape(
        10,
        3,
        1
    )
)

signature_behaviour = (
    signature_profiles[:, 3:12]
    .reshape(
        10,
        3,
        3
    )
)

signature_process = (
    signature_profiles[:, 12:21]
    .reshape(
        10,
        3,
        3
    )
)


signature_token_profiles = np.concatenate(
    [
        signature_credit,
        signature_behaviour,
        signature_process
    ],
    axis=2
).astype(
    np.float32
)


# Round-trip integrity check

roundtrip_profiles = np.concatenate(
    [
        signature_token_profiles[:, :, 0]
        .reshape(10, 3),

        signature_token_profiles[:, :, 1:4]
        .reshape(10, 9),

        signature_token_profiles[:, :, 4:7]
        .reshape(10, 9)
    ],
    axis=1
)


assert np.allclose(
    roundtrip_profiles,
    signature_profiles,
    atol=1e-7
)


print(
    "Signature token profiles:",
    signature_token_profiles.shape
)

print(
    "Per-process feature order:"
)

print(
    "[credit_residual, "
    "scorable_residual, "
    "no_response_residual, "
    "not_reached_residual, "
    "process_1_residual, "
    "process_2_residual, "
    "process_3_residual]"
)

print(
    "\n21-D -> 3x7 calibration mapping: PASS"
)

Signature token profiles: (10, 3, 7)
Per-process feature order:
[credit_residual, scorable_residual, no_response_residual, not_reached_residual, process_1_residual, process_2_residual, process_3_residual]

21-D -> 3x7 calibration mapping: PASS


In [ ]:
# Compact leakage-safe Fold-0 preprocessing

common_student_vars = list(
    CONFIG["common_student_vars"]
)

common_school_vars = list(
    CONFIG["common_school_vars"]
)


class ContextPreprocessor:

    def __init__(
        self,
        registry,
        variables
    ):

        reg = registry[
            registry["variable"].isin(
                variables
            )
        ].copy()

        self.continuous_vars = (
            reg.loc[
                reg["feature_type"]
                == "continuous",
                "variable"
            ]
            .tolist()
        )

        self.categorical_vars = (
            reg.loc[
                reg["feature_type"].isin(
                    [
                        "categorical",
                        "binary",
                        "ordinal"
                    ]
                ),
                "variable"
            ]
            .tolist()
        )

        self.medians = {}
        self.iqrs = {}
        self.category_maps = {}


    def fit(
        self,
        df,
        indices
    ):

        train_df = df.iloc[
            indices
        ]


        for var in self.continuous_vars:

            x = pd.to_numeric(
                train_df[var],
                errors="coerce"
            )

            med = x.median()
            iqr = (
                x.quantile(0.75)
                -
                x.quantile(0.25)
            )

            if (
                not np.isfinite(iqr)
                or iqr < 1e-8
            ):
                iqr = 1.0

            self.medians[var] = float(med)
            self.iqrs[var] = float(iqr)


        for var in self.categorical_vars:

            values = sorted(
                train_df[var]
                .dropna()
                .unique()
                .tolist(),
                key=str
            )

            # 0 = missing
            # 1 = unseen
            # 2+ = observed categories
            self.category_maps[var] = {
                value: i + 2
                for i, value
                in enumerate(values)
            }


        return self


    def transform(
        self,
        df
    ):

        n = len(df)

        cont = []
        cont_mask = []


        for var in self.continuous_vars:

            x = pd.to_numeric(
                df[var],
                errors="coerce"
            ).to_numpy(
                dtype=np.float32
            )

            mask = np.isfinite(x)

            z = (
                x
                -
                self.medians[var]
            ) / self.iqrs[var]

            z[
                ~mask
            ] = 0.0

            cont.append(z)
            cont_mask.append(mask)


        continuous = (
            np.column_stack(cont)
            if cont
            else np.empty(
                (n, 0),
                dtype=np.float32
            )
        )

        continuous_mask = (
            np.column_stack(cont_mask)
            if cont_mask
            else np.empty(
                (n, 0),
                dtype=bool
            )
        )


        cat = []


        for var in self.categorical_vars:

            mapping = (
                self.category_maps[var]
            )

            values = df[var].to_numpy()

            encoded = np.zeros(
                n,
                dtype=np.int64
            )


            for i, value in enumerate(
                values
            ):

                if pd.isna(value):
                    encoded[i] = 0
                else:
                    encoded[i] = (
                        mapping.get(
                            value,
                            1
                        )
                    )


            cat.append(encoded)


        categorical = (
            np.column_stack(cat)
            if cat
            else np.empty(
                (n, 0),
                dtype=np.int64
            )
        )


        return {
            "continuous":
                continuous.astype(
                    np.float32
                ),

            "continuous_mask":
                continuous_mask,

            "categorical":
                categorical
        }


    def cardinalities(
        self
    ):

        return [
            len(
                self.category_maps[var]
            ) + 2

            for var
            in self.categorical_vars
        ]


class ProcessPreprocessor:

    def fit(
        self,
        x
    ):

        self.median = np.nanmedian(
            x,
            axis=0
        )

        q25 = np.nanpercentile(
            x,
            25,
            axis=0
        )

        q75 = np.nanpercentile(
            x,
            75,
            axis=0
        )

        self.iqr = (
            q75 - q25
        )


        self.median[
            ~np.isfinite(
                self.median
            )
        ] = 0.0

        self.iqr[
            ~np.isfinite(
                self.iqr
            )
        ] = 1.0

        self.iqr[
            self.iqr < 1e-8
        ] = 1.0


        return self


    def transform(
        self,
        x
    ):

        mask = np.isfinite(
            x
        )

        scaled = (
            x
            -
            self.median[
                None, :, :
            ]
        ) / self.iqr[
            None, :, :
        ]


        scaled = np.asarray(
            scaled,
            dtype=np.float32
        )

        scaled[
            ~mask
        ] = 0.0


        return (
            scaled,
            mask
        )


student_pp = (
    ContextPreprocessor(
        context_registry_locked,
        common_student_vars
    )
    .fit(
        model_context,
        train_idx
    )
)

school_pp = (
    ContextPreprocessor(
        context_registry_locked,
        common_school_vars
    )
    .fit(
        model_context,
        train_idx
    )
)

process_pp = (
    ProcessPreprocessor()
    .fit(
        process_tensor[
            train_idx
        ]
    )
)


print(
    "Student continuous:",
    student_pp.continuous_vars
)

print(
    "Student categorical:",
    student_pp.categorical_vars
)

print(
    "School continuous:",
    school_pp.continuous_vars
)

print(
    "School categorical:",
    school_pp.categorical_vars
)

print(
    "\nFold-0 train-only preprocessors ready."
)

Student continuous: ['AGE', 'HOMEPOS']
Student categorical: ['ST004D01T', 'REPEAT', 'IMMIG', 'HOME_TEST_LANG_MATCH']
School continuous: []
School categorical: ['SC001Q01TA', 'SRESPCUR', 'SRESPRES']

Fold-0 train-only preprocessors ready.


In [ ]:
# Build final Fold-0 dataset bundle

train_countries = sorted(
    primary_meta.iloc[
        train_idx
    ]["CNT"]
    .unique()
    .tolist()
)


country_to_train_id = {
    cnt: i
    for i, cnt
    in enumerate(
        train_countries
    )
}


def make_split(
    indices
):

    student_ctx = (
        student_pp.transform(
            model_context.iloc[
                indices
            ]
        )
    )

    school_ctx = (
        school_pp.transform(
            model_context.iloc[
                indices
            ]
        )
    )


    # Raw/log1p process values for calibrated branch

    process_raw = np.asarray(
        process_tensor[
            indices
        ],
        dtype=np.float32
    )


    # Fold-scaled process values for main encoder

    process_values, process_mask = (
        process_pp.transform(
            process_raw
        )
    )


    countries = (
        primary_meta.iloc[
            indices
        ]["CNT"]
        .to_numpy()
    )


    country_ids = np.array(
        [
            country_to_train_id.get(
                cnt,
                -1
            )
            for cnt
            in countries
        ],
        dtype=np.int64
    )


    booklet_ids = np.asarray(
        booklet_nuisance_ids[
            indices
        ],
        dtype=np.int64
    )


    signature_ids = (
        BOOKLET_TO_SIGNATURE[
            booklet_ids
        ]
        .astype(
            np.int64
        )
    )


    return {

        "credit_tokens":
            np.asarray(
                credit_tokens[
                    indices
                ],
                dtype=np.int64
            ),

        "behaviour_tokens":
            np.asarray(
                behaviour_tokens[
                    indices
                ],
                dtype=np.int64
            ),

        "performance_mask":
            np.asarray(
                performance_mask[
                    indices
                ],
                dtype=bool
            ),

        "interaction_mask":
            np.asarray(
                interaction_mask[
                    indices
                ],
                dtype=bool
            ),

        "design_mask":
            np.asarray(
                student_design_mask[
                    indices
                ],
                dtype=bool
            ),

        "process_raw":
            process_raw,

        "process_values":
            process_values,

        "process_mask":
            process_mask,

        "student_continuous":
            student_ctx[
                "continuous"
            ],

        "student_continuous_mask":
            student_ctx[
                "continuous_mask"
            ],

        "student_categorical":
            student_ctx[
                "categorical"
            ],

        "school_continuous":
            school_ctx[
                "continuous"
            ],

        "school_continuous_mask":
            school_ctx[
                "continuous_mask"
            ],

        "school_categorical":
            school_ctx[
                "categorical"
            ],

        "booklet_nuisance_id":
            booklet_ids,

        "design_signature_id":
            signature_ids,

        "country_train_id":
            country_ids,

        "CNT":
            countries
    }


fold0 = {

    "train":
        make_split(
            train_idx
        ),

    "validation":
        make_split(
            validation_idx
        ),

    "test":
        make_split(
            test_idx
        )
}


print(
    "Train:",
    len(
        fold0["train"]["CNT"]
    )
)

print(
    "Validation:",
    len(
        fold0["validation"]["CNT"]
    )
)

print(
    "Test:",
    len(
        fold0["test"]["CNT"]
    )
)

print(
    "\nScaled process:",
    fold0[
        "train"
    ][
        "process_values"
    ].shape
)

print(
    "Raw process:",
    fold0[
        "train"
    ][
        "process_raw"
    ].shape
)

print(
    "Design signatures:",
    len(
        np.unique(
            fold0[
                "train"
            ][
                "design_signature_id"
            ]
        )
    )
)

Train: 99966
Validation: 20949
Test: 21649

Scaled process: (99966, 32, 3)
Raw process: (99966, 32, 3)
Design signatures: 10


In [ ]:
# Dataset and country-balanced DataLoader

class CReSTFinalDataset(Dataset):

    def __init__(
        self,
        split_data
    ):
        self.d = split_data


    def __len__(
        self
    ):
        return len(
            self.d["CNT"]
        )


    def __getitem__(
        self,
        idx
    ):

        tensor_specs = {

            "credit_tokens":
                torch.long,

            "behaviour_tokens":
                torch.long,

            "performance_mask":
                torch.bool,

            "interaction_mask":
                torch.bool,

            "design_mask":
                torch.bool,

            "process_raw":
                torch.float32,

            "process_values":
                torch.float32,

            "process_mask":
                torch.bool,

            "student_continuous":
                torch.float32,

            "student_continuous_mask":
                torch.bool,

            "student_categorical":
                torch.long,

            "school_continuous":
                torch.float32,

            "school_continuous_mask":
                torch.bool,

            "school_categorical":
                torch.long,

            "booklet_nuisance_id":
                torch.long,

            "design_signature_id":
                torch.long,

            "country_train_id":
                torch.long
        }


        return {
            key:
                torch.as_tensor(
                    self.d[key][idx],
                    dtype=dtype
                )

            for key, dtype
            in tensor_specs.items()
        }


train_dataset = CReSTFinalDataset(
    fold0["train"]
)

validation_dataset = CReSTFinalDataset(
    fold0["validation"]
)

test_dataset = CReSTFinalDataset(
    fold0["test"]
)


# Country-balanced sampling

train_country_array = (
    fold0["train"]["CNT"]
)

unique_train_countries, country_counts = (
    np.unique(
        train_country_array,
        return_counts=True
    )
)


country_count_map = {
    cnt: count

    for cnt, count
    in zip(
        unique_train_countries,
        country_counts
    )
}


sampling_weights = np.array(
    [
        1.0
        / country_count_map[cnt]

        for cnt
        in train_country_array
    ],
    dtype=np.float64
)


BATCH_SIZE = int(
    CONFIG["batch_size"]
)


train_sampler = WeightedRandomSampler(

    weights=torch.as_tensor(
        sampling_weights,
        dtype=torch.double
    ),

    num_samples=len(
        train_dataset
    ),

    replacement=True,

    generator=(
        torch.Generator()
        .manual_seed(
            GLOBAL_SEED + 700
        )
    )
)


train_loader = DataLoader(

    train_dataset,

    batch_size=BATCH_SIZE,

    sampler=train_sampler,

    shuffle=False,

    num_workers=0,

    pin_memory=torch.cuda.is_available()
)


validation_loader = DataLoader(

    validation_dataset,

    batch_size=BATCH_SIZE,

    shuffle=False,

    num_workers=0
)


print(
    "Batch size:",
    BATCH_SIZE
)

print(
    "Train batches:",
    len(train_loader)
)

print(
    "Validation batches:",
    len(validation_loader)
)

print(
    "Training systems:",
    len(
        unique_train_countries
    )
)

Batch size: 256
Train batches: 391
Validation batches: 82
Training systems: 45


In [ ]:
# Locked calibration tensors

credit_item_mean_t = torch.as_tensor(
    credit_item_mean,
    dtype=torch.float32,
    device=device
)

behaviour_item_rates_t = torch.as_tensor(
    behaviour_item_rates,
    dtype=torch.float32,
    device=device
)

process_item_mean_t = torch.as_tensor(
    process_item_mean,
    dtype=torch.float32,
    device=device
)

process_item_std_t = torch.as_tensor(
    process_item_std,
    dtype=torch.float32,
    device=device
)

signature_token_profiles_t = torch.as_tensor(
    signature_token_profiles,
    dtype=torch.float32,
    device=device
)

item_ideation_ids_t = torch.as_tensor(
    item_ideation_ids,
    dtype=torch.long,
    device=device
)


print(
    "Credit baseline:",
    tuple(
        credit_item_mean_t.shape
    )
)

print(
    "Behaviour baseline:",
    tuple(
        behaviour_item_rates_t.shape
    )
)

print(
    "Process baseline:",
    tuple(
        process_item_mean_t.shape
    )
)

print(
    "Signature token profiles:",
    tuple(
        signature_token_profiles_t.shape
    )
)

Credit baseline: (32,)
Behaviour baseline: (32, 3)
Process baseline: (32, 3)
Signature token profiles: (10, 3, 7)


In [ ]:
# Final calibrated general-token builder

def masked_feature_mean(
    values,
    mask,
    dim=1
):

    mask_f = mask.to(
        values.dtype
    )

    numerator = (
        values
        * mask_f
    ).sum(
        dim=dim
    )

    denominator = (
        mask_f.sum(
            dim=dim
        )
        .clamp_min(
            1.0
        )
    )

    return (
        numerator
        /
        denominator
    )


def build_calibrated_general_tokens(
    batch
):

    credit = batch[
        "credit_tokens"
    ]

    behaviour = batch[
        "behaviour_tokens"
    ]

    performance = batch[
        "performance_mask"
    ]

    interaction = batch[
        "interaction_mask"
    ]

    process_raw = batch[
        "process_raw"
    ]

    process_mask = batch[
        "process_mask"
    ]

    design_signature = batch[
        "design_signature_id"
    ]


    process_tokens = []


    # One summary token per GDI / GCI / EII

    for process_id in range(3):

        item_selector = (
            item_ideation_ids_t
            == process_id
        )


        # 1. Credit residual
        #
        # SSL MASK token = 4 and is explicitly excluded.

        c = credit[
            :,
            item_selector
        ]


        c_valid = (

            performance[
                :,
                item_selector
            ]

            &

            (c >= 1)

            &

            (c <= 3)
        )


        c_score = (
            c.float()
            - 1.0
        ) / 2.0


        c_residual = (

            c_score

            -

            credit_item_mean_t[
                item_selector
            ]
            .unsqueeze(0)
        )


        c_residual = torch.where(

            c_valid,

            c_residual,

            torch.zeros_like(
                c_residual
            )
        )


        credit_summary = (
            masked_feature_mean(
                c_residual,
                c_valid,
                dim=1
            )
        )


        # 2. Behaviour residuals
        #
        # scorable / no-response / not-reached

        b = behaviour[
            :,
            item_selector
        ]


        b_valid = (

            interaction[
                :,
                item_selector
            ]

            &

            (b >= 1)

            &

            (b <= 3)
        )


        behaviour_summaries = []


        for behaviour_index, token in enumerate(
            [1, 2, 3]
        ):

            indicator = (
                b == token
            ).float()


            residual = (

                indicator

                -

                behaviour_item_rates_t[
                    item_selector,
                    behaviour_index
                ]
                .unsqueeze(0)
            )


            residual = torch.where(

                b_valid,

                residual,

                torch.zeros_like(
                    residual
                )
            )


            summary = masked_feature_mean(
                residual,
                b_valid,
                dim=1
            )


            behaviour_summaries.append(
                summary
            )


        # 3. Process residuals
        #
        # IMPORTANT:
        # process_mask determines availability.
        # Therefore SSL-masked process cells are excluded
        # even though process_raw retains the original value.

        p_raw = process_raw[
            :,
            item_selector,
            :
        ]


        p_valid = process_mask[
            :,
            item_selector,
            :
        ]


        item_mean = (
            process_item_mean_t[
                item_selector,
                :
            ]
            .unsqueeze(0)
        )


        item_std = (
            process_item_std_t[
                item_selector,
                :
            ]
            .unsqueeze(0)
        )


        p_safe = torch.nan_to_num(
            p_raw,
            nan=0.0,
            posinf=0.0,
            neginf=0.0
        )


        p_residual = (

            p_safe
            -
            item_mean

        ) / (

            item_std
            +
            1e-6
        )


        p_residual = torch.where(

            p_valid,

            p_residual,

            torch.zeros_like(
                p_residual
            )
        )


        process_summary = (
            masked_feature_mean(
                p_residual,
                p_valid,
                dim=1
            )
        )


        # Assemble 7-D token

        token_features = torch.stack(
            [
                credit_summary,

                behaviour_summaries[0],
                behaviour_summaries[1],
                behaviour_summaries[2],

                process_summary[:, 0],
                process_summary[:, 1],
                process_summary[:, 2]
            ],
            dim=1
        )


        process_tokens.append(
            token_features
        )


    # [B, 3, 7]
    tokens = torch.stack(
        process_tokens,
        dim=1
    )


    # Country-equal assessment-design calibration
    #
    # Signature ID is used ONLY to select a fixed,
    # training-derived nuisance offset.
    # It is not embedded or passed as a predictive feature.

    nuisance_offset = (
        signature_token_profiles_t[
            design_signature
        ]
    )


    calibrated_tokens = (
        tokens
        -
        nuisance_offset
    )


    # Process availability

    availability = []


    for process_id in range(3):

        selector = (
            item_ideation_ids_t
            == process_id
        )


        available = (

            interaction[
                :,
                selector
            ].any(
                dim=1
            )
        )


        availability.append(
            available
        )


    availability = torch.stack(
        availability,
        dim=1
    )


    return (
        calibrated_tokens,
        availability
    )


print(
    "Final calibrated general-token builder ready."
)

Final calibrated general-token builder ready.


In [ ]:
# Smoke-test calibrated general tokens

batch = next(
    iter(
        train_loader
    )
)


batch = {
    key:
        value[:64]
        .to(device)

    for key, value
    in batch.items()
}


with torch.no_grad():

    general_tokens, general_available = (
        build_calibrated_general_tokens(
            batch
        )
    )


print(
    "General tokens:",
    tuple(
        general_tokens.shape
    )
)

print(
    "Availability:",
    tuple(
        general_available.shape
    )
)

print(
    "All finite:",
    bool(
        torch.isfinite(
            general_tokens
        ).all()
    )
)

print(
    "Students with all 3 processes:",
    int(
        general_available.all(
            dim=1
        ).sum()
    ),
    "/",
    len(
        general_available
    )
)


assert (
    general_tokens.shape
    == (64, 3, 7)
)

assert (
    general_available.shape
    == (64, 3)
)

assert torch.isfinite(
    general_tokens
).all()


print(
    "\nCalibrated-token smoke test: PASS"
)

General tokens: (64, 3, 7)
Availability: (64, 3)
All finite: True
Students with all 3 processes: 63 / 64

Calibrated-token smoke test: PASS


In [ ]:
# Final CReST utilities and context cardinalities

# Context categorical vocabularies
#
# Base IDs:
#   0 = missing
#   1 = unseen
#   2+ = observed category
#
# Encoder receives one extra SSL MASK token.

student_base_cardinalities = (
    student_pp.cardinalities()
)

school_base_cardinalities = (
    school_pp.cardinalities()
)

student_context_mask_ids = (
    student_base_cardinalities.copy()
)

school_context_mask_ids = (
    school_base_cardinalities.copy()
)

student_model_cardinalities = [
    c + 1
    for c in student_base_cardinalities
]

school_model_cardinalities = [
    c + 1
    for c in school_base_cardinalities
]


# Gradient reversal

class GradientReversalFunction(
    torch.autograd.Function
):

    @staticmethod
    def forward(
        ctx,
        x,
        lambda_grl
    ):
        ctx.lambda_grl = lambda_grl
        return x.view_as(x)


    @staticmethod
    def backward(
        ctx,
        grad_output
    ):
        return (
            -ctx.lambda_grl
            * grad_output,
            None
        )


def grad_reverse(
    x,
    lambda_grl=1.0
):
    return GradientReversalFunction.apply(
        x,
        lambda_grl
    )


# Masked mean

def masked_mean(
    x,
    mask,
    dim=1,
    eps=1e-8
):

    w = (
        mask
        .unsqueeze(-1)
        .to(x.dtype)
    )

    return (
        (x * w).sum(dim=dim)
        /
        w.sum(dim=dim)
        .clamp_min(eps)
    )


# Context encoder

class ContextEncoder(nn.Module):

    def __init__(
        self,
        n_continuous,
        categorical_cardinalities,
        d_model=128,
        cat_embedding_dim=16,
        dropout=0.1
    ):

        super().__init__()

        self.n_continuous = (
            n_continuous
        )

        self.categorical_embeddings = (
            nn.ModuleList(
                [
                    nn.Embedding(
                        cardinality,
                        cat_embedding_dim
                    )
                    for cardinality
                    in categorical_cardinalities
                ]
            )
        )


        input_dim = (
            2 * n_continuous
            +
            len(
                categorical_cardinalities
            )
            * cat_embedding_dim
        )


        self.encoder = nn.Sequential(

            nn.Linear(
                input_dim,
                d_model
            ),

            nn.GELU(),

            nn.LayerNorm(
                d_model
            ),

            nn.Dropout(
                dropout
            ),

            nn.Linear(
                d_model,
                d_model
            ),

            nn.GELU(),

            nn.LayerNorm(
                d_model
            )
        )


    def forward(
        self,
        continuous,
        continuous_mask,
        categorical
    ):

        parts = []


        if self.n_continuous > 0:

            parts.extend(
                [
                    continuous,
                    continuous_mask.float()
                ]
            )


        for j, embedding in enumerate(
            self.categorical_embeddings
        ):

            parts.append(
                embedding(
                    categorical[:, j]
                )
            )


        x = torch.cat(
            parts,
            dim=1
        )


        return self.encoder(
            x
        )


print(
    "Student base cardinalities:",
    student_base_cardinalities
)

print(
    "School base cardinalities:",
    school_base_cardinalities
)

print(
    "Final utility layers ready."
)

Student base cardinalities: [4, 4, 5, 4]
School base cardinalities: [8, 7, 9]
Final utility layers ready.


In [ ]:
# Final CReST architecture

class CReSTFinal(nn.Module):

    def __init__(
        self,
        item_domain_ids,
        item_ideation_ids,

        student_n_continuous,
        student_cat_cardinalities,

        school_n_continuous,
        school_cat_cardinalities,

        n_train_countries=45,
        n_booklets=30,
        n_design_signatures=10,

        n_domains=4,
        n_processes=3,

        d_model=128,
        latent_dim=64,

        n_heads=8,
        n_layers=3,
        dropout=0.1
    ):

        super().__init__()


        self.n_items = 32
        self.n_domains = n_domains
        self.n_processes = n_processes

        self.d_model = d_model
        self.latent_dim = latent_dim


        # Fixed item metadata

        self.register_buffer(
            "item_domain_ids",
            torch.as_tensor(
                np.asarray(
                    item_domain_ids
                ),
                dtype=torch.long
            )
        )

        self.register_buffer(
            "item_ideation_ids",
            torch.as_tensor(
                np.asarray(
                    item_ideation_ids
                ),
                dtype=torch.long
            )
        )


        # Embeddings
        #
        # Exact item identity remains decoder-only.

        self.item_embedding = nn.Embedding(
            self.n_items,
            d_model
        )

        self.domain_embedding = nn.Embedding(
            n_domains,
            d_model
        )

        self.ideation_embedding = nn.Embedding(
            n_processes,
            d_model
        )

        # 0 unavailable
        # 1 no credit
        # 2 partial
        # 3 full
        # 4 SSL mask
        self.credit_embedding = nn.Embedding(
            5,
            d_model
        )

        # 0 unavailable
        # 1 scorable
        # 2 no response
        # 3 not reached
        self.behaviour_embedding = nn.Embedding(
            4,
            d_model
        )


        # Main item-level process projection

        self.process_projection = nn.Sequential(

            nn.Linear(
                6,
                d_model
            ),

            nn.GELU(),

            nn.LayerNorm(
                d_model
            )
        )


        # Performance Transformer

        performance_layer = (
            nn.TransformerEncoderLayer(

                d_model=d_model,
                nhead=n_heads,
                dim_feedforward=4 * d_model,

                dropout=dropout,
                activation="gelu",

                batch_first=True,
                norm_first=True
            )
        )


        self.performance_encoder = (
            nn.TransformerEncoder(

                performance_layer,

                num_layers=n_layers,

                norm=nn.LayerNorm(
                    d_model
                )
            )
        )


        # Process Transformer

        process_layer = (
            nn.TransformerEncoderLayer(

                d_model=d_model,
                nhead=n_heads,
                dim_feedforward=4 * d_model,

                dropout=dropout,
                activation="gelu",

                batch_first=True,
                norm_first=True
            )
        )


        self.process_encoder = (
            nn.TransformerEncoder(

                process_layer,

                num_layers=n_layers,

                norm=nn.LayerNorm(
                    d_model
                )
            )
        )


        # Cross-view fusion

        self.cross_attention = (
            nn.MultiheadAttention(

                embed_dim=d_model,
                num_heads=n_heads,

                dropout=dropout,

                batch_first=True
            )
        )


        self.cross_norm = nn.LayerNorm(
            d_model
        )


        # FINAL DESIGN-ROBUST GENERAL BRANCH
        #
        # 7 calibrated features -> d_model
        # then Transformer over exactly 3 tokens:
        # GDI / GCI / EII

        self.general_token_projection = (
            nn.Sequential(

                nn.Linear(
                    7,
                    d_model
                ),

                nn.GELU(),

                nn.LayerNorm(
                    d_model
                )
            )
        )


        general_layer = (
            nn.TransformerEncoderLayer(

                d_model=d_model,
                nhead=n_heads,

                dim_feedforward=(
                    2 * d_model
                ),

                dropout=dropout,
                activation="gelu",

                batch_first=True,
                norm_first=True
            )
        )


        self.general_process_encoder = (
            nn.TransformerEncoder(

                general_layer,

                num_layers=1,

                norm=nn.LayerNorm(
                    d_model
                )
            )
        )


        self.general_pool_norm = (
            nn.LayerNorm(
                d_model
            )
        )


        self.creative_projection = (
            nn.Sequential(

                nn.Linear(
                    d_model,
                    latent_dim
                ),

                nn.LayerNorm(
                    latent_dim
                )
            )
        )


        # Context branch

        self.student_encoder = ContextEncoder(

            n_continuous=(
                student_n_continuous
            ),

            categorical_cardinalities=(
                student_cat_cardinalities
            ),

            d_model=d_model,

            dropout=dropout
        )


        self.school_encoder = ContextEncoder(

            n_continuous=(
                school_n_continuous
            ),

            categorical_cardinalities=(
                school_cat_cardinalities
            ),

            d_model=d_model,

            dropout=dropout
        )


        self.context_fusion = nn.Sequential(

            nn.Linear(
                2 * d_model,
                d_model
            ),

            nn.GELU(),

            nn.LayerNorm(
                d_model
            )
        )


        self.context_projection = (
            nn.Sequential(

                nn.Linear(
                    d_model,
                    latent_dim
                ),

                nn.LayerNorm(
                    latent_dim
                )
            )
        )


        # Domain-specific branch

        self.domain_shared_projection = (
            nn.Sequential(

                nn.Linear(
                    d_model,
                    latent_dim
                ),

                nn.GELU(),

                nn.LayerNorm(
                    latent_dim
                )
            )
        )


        self.domain_specific_projections = (
            nn.ModuleList(
                [
                    nn.Sequential(

                        nn.Linear(
                            d_model,
                            latent_dim
                        ),

                        nn.GELU(),

                        nn.LayerNorm(
                            latent_dim
                        )
                    )

                    for _ in range(
                        n_domains
                    )
                ]
            )
        )


        # Item reconstruction heads

        self.credit_head = nn.Linear(
            d_model,
            3
        )

        self.process_head = nn.Linear(
            d_model,
            3
        )


        # Domain decomposition decoder

        self.domain_latent_decoder = (
            nn.Sequential(

                nn.Linear(
                    latent_dim,
                    d_model
                ),

                nn.GELU(),

                nn.LayerNorm(
                    d_model
                )
            )
        )


        self.decomposition_credit_head = (
            nn.Linear(
                d_model,
                3
            )
        )


        # Context reconstruction

        total_continuous = (
            student_n_continuous
            +
            school_n_continuous
        )


        self.context_continuous_head = (
            nn.Linear(
                latent_dim,
                total_continuous
            )
            if total_continuous > 0
            else None
        )


        base_context_cardinalities = (

            student_base_cardinalities

            +

            school_base_cardinalities
        )


        self.context_categorical_heads = (
            nn.ModuleList(
                [
                    nn.Linear(
                        latent_dim,
                        cardinality
                    )

                    for cardinality
                    in base_context_cardinalities
                ]
            )
        )


        # Nuisance routing

        self.booklet_classifier = nn.Sequential(

            nn.Linear(
                latent_dim,
                latent_dim
            ),

            nn.GELU(),

            nn.Linear(
                latent_dim,
                n_booklets
            )
        )


        self.design_signature_classifier = (
            nn.Sequential(

                nn.Linear(
                    latent_dim,
                    latent_dim
                ),

                nn.GELU(),

                nn.Linear(
                    latent_dim,
                    n_design_signatures
                )
            )
        )


        self.context_country_classifier = (
            nn.Sequential(

                nn.Linear(
                    latent_dim,
                    latent_dim
                ),

                nn.GELU(),

                nn.Linear(
                    latent_dim,
                    n_train_countries
                )
            )
        )


        self.creative_country_classifier = (
            nn.Sequential(

                nn.Linear(
                    latent_dim,
                    latent_dim
                ),

                nn.GELU(),

                nn.Linear(
                    latent_dim,
                    n_train_countries
                )
            )
        )


    # Item encoder metadata
    #
    # NO exact item identity.

    def encoder_metadata(
        self,
        batch_size
    ):

        meta = (

            self.domain_embedding(
                self.item_domain_ids
            )

            +

            self.ideation_embedding(
                self.item_ideation_ids
            )
        )


        return (
            meta
            .unsqueeze(0)
            .expand(
                batch_size,
                -1,
                -1
            )
        )


    # Decoder metadata
    #
    # Exact item identity is allowed only here.

    def decoder_metadata(
        self,
        batch_size,
        device
    ):

        item_ids = torch.arange(
            self.n_items,
            device=device
        )


        meta = (

            self.item_embedding(
                item_ids
            )

            +

            self.domain_embedding(
                self.item_domain_ids
            )

            +

            self.ideation_embedding(
                self.item_ideation_ids
            )
        )


        return (
            meta
            .unsqueeze(0)
            .expand(
                batch_size,
                -1,
                -1
            )
        )


    # Forward

    def forward(
        self,
        batch,
        lambda_booklet_grl=1.0,
        lambda_design_grl=1.0,
        lambda_country_grl=1.0
    ):

        credit = batch[
            "credit_tokens"
        ]

        behaviour = batch[
            "behaviour_tokens"
        ]

        interaction_mask = batch[
            "interaction_mask"
        ]

        process_values = batch[
            "process_values"
        ]

        process_mask = batch[
            "process_mask"
        ]


        batch_size = (
            credit.shape[0]
        )

        device_local = (
            credit.device
        )


        # A. ITEM-LEVEL PERFORMANCE BRANCH

        encoder_meta = (
            self.encoder_metadata(
                batch_size
            )
        )


        performance_tokens = (

            encoder_meta

            +

            self.credit_embedding(
                credit
            )

            +

            self.behaviour_embedding(
                behaviour
            )
        )


        performance_encoded = (
            self.performance_encoder(

                performance_tokens,

                src_key_padding_mask=(
                    ~interaction_mask
                )
            )
        )


        # B. ITEM-LEVEL PROCESS BRANCH

        process_input = torch.cat(
            [
                process_values,
                process_mask.float()
            ],
            dim=-1
        )


        process_tokens = (

            encoder_meta

            +

            self.process_projection(
                process_input
            )
        )


        process_item_mask = (
            process_mask.any(
                dim=-1
            )
        )


        safe_process_mask = (
            process_item_mask.clone()
        )


        no_process = (
            ~safe_process_mask.any(
                dim=1
            )
        )


        if no_process.any():

            safe_process_mask[
                no_process,
                0
            ] = True


            process_tokens = (
                process_tokens.clone()
            )


            process_tokens[
                no_process,
                0,
                :
            ] = 0.0


        process_encoded = (
            self.process_encoder(

                process_tokens,

                src_key_padding_mask=(
                    ~safe_process_mask
                )
            )
        )


        # C. CROSS-VIEW ITEM REPRESENTATION

        cross_output, _ = (
            self.cross_attention(

                query=performance_encoded,

                key=process_encoded,

                value=process_encoded,

                key_padding_mask=(
                    ~safe_process_mask
                ),

                need_weights=False
            )
        )


        if no_process.any():

            cross_output = (
                cross_output.clone()
            )

            cross_output[
                no_process
            ] = 0.0


        creative_item_tokens = (
            self.cross_norm(

                performance_encoded

                +

                cross_output
            )
        )


        # D. FINAL DESIGN-ROBUST GENERAL BRANCH

        (
            calibrated_general_tokens,
            general_available
        ) = build_calibrated_general_tokens(
            batch
        )


        general_tokens = (
            self.general_token_projection(
                calibrated_general_tokens
            )
        )


        # Add only ideation-process identity.
        process_ids = torch.arange(
            self.n_processes,
            device=device_local
        )


        general_tokens = (

            general_tokens

            +

            self.ideation_embedding(
                process_ids
            )
            .unsqueeze(0)
        )


        safe_general_available = (
            general_available.clone()
        )


        no_general = (
            ~safe_general_available.any(
                dim=1
            )
        )


        if no_general.any():

            safe_general_available[
                no_general,
                0
            ] = True


            general_tokens = (
                general_tokens.clone()
            )


            general_tokens[
                no_general,
                0,
                :
            ] = 0.0


        general_encoded = (
            self.general_process_encoder(

                general_tokens,

                src_key_padding_mask=(
                    ~safe_general_available
                )
            )
        )


        weights = (
            general_available
            .float()
            .unsqueeze(-1)
        )


        general_pooled = (

            (
                general_encoded
                * weights
            )
            .sum(
                dim=1
            )

            /

            weights.sum(
                dim=1
            )
            .clamp_min(
                1.0
            )
        )


        general_pooled = (
            self.general_pool_norm(
                general_pooled
            )
        )


        z_general = (
            self.creative_projection(
                general_pooled
            )
        )


        # E. CONTEXT REPRESENTATION

        student_context = (
            self.student_encoder(

                batch[
                    "student_continuous"
                ],

                batch[
                    "student_continuous_mask"
                ],

                batch[
                    "student_categorical"
                ]
            )
        )


        school_context = (
            self.school_encoder(

                batch[
                    "school_continuous"
                ],

                batch[
                    "school_continuous_mask"
                ],

                batch[
                    "school_categorical"
                ]
            )
        )


        context_fused = (
            self.context_fusion(

                torch.cat(
                    [
                        student_context,
                        school_context
                    ],
                    dim=1
                )
            )
        )


        z_context = (
            self.context_projection(
                context_fused
            )
        )


        # F. DOMAIN-SPECIFIC REPRESENTATIONS

        domain_pooled = []
        domain_available = []


        for domain_id in range(
            self.n_domains
        ):

            selector = (

                self.item_domain_ids
                .unsqueeze(0)

                == domain_id
            )


            valid = (

                interaction_mask

                &

                selector
            )


            available = (
                valid.any(
                    dim=1
                )
            )


            pooled = masked_mean(

                creative_item_tokens,

                valid,

                dim=1
            )


            pooled = torch.where(

                available.unsqueeze(-1),

                pooled,

                torch.zeros_like(
                    pooled
                )
            )


            domain_pooled.append(
                pooled
            )

            domain_available.append(
                available
            )


        domain_pooled = torch.stack(
            domain_pooled,
            dim=1
        )


        domain_available = torch.stack(
            domain_available,
            dim=1
        )


        z_domain_shared = (
            self.domain_shared_projection(
                domain_pooled
            )
        )


        z_domain_shared = torch.where(

            domain_available.unsqueeze(-1),

            z_domain_shared,

            torch.zeros_like(
                z_domain_shared
            )
        )


        specific_list = []


        for domain_id in range(
            self.n_domains
        ):

            z_specific = (
                self.domain_specific_projections[
                    domain_id
                ](
                    domain_pooled[
                        :,
                        domain_id,
                        :
                    ]
                )
            )


            z_specific = torch.where(

                domain_available[
                    :,
                    domain_id
                ]
                .unsqueeze(-1),

                z_specific,

                torch.zeros_like(
                    z_specific
                )
            )


            specific_list.append(
                z_specific
            )


        z_domain_specific = torch.stack(
            specific_list,
            dim=1
        )


        z_domain_creative = (

            z_general.unsqueeze(1)

            +

            z_domain_specific
        )


        z_domain_creative = torch.where(

            domain_available.unsqueeze(-1),

            z_domain_creative,

            torch.zeros_like(
                z_domain_creative
            )
        )


        # G. ITEM RECONSTRUCTION

        decoder_meta = (
            self.decoder_metadata(
                batch_size,
                device_local
            )
        )


        decoder_item_tokens = (

            creative_item_tokens

            +

            decoder_meta
        )


        credit_logits = (
            self.credit_head(
                decoder_item_tokens
            )
        )


        process_prediction = (
            self.process_head(
                decoder_item_tokens
            )
        )


        # H. DOMAIN-DECOMPOSITION RECONSTRUCTION

        domain_latent_per_item = (
            z_domain_creative
            .index_select(

                dim=1,

                index=self.item_domain_ids
            )
        )


        decomposition_tokens = (

            self.domain_latent_decoder(
                domain_latent_per_item
            )

            +

            decoder_meta
        )


        decomposition_credit_logits = (
            self.decomposition_credit_head(
                decomposition_tokens
            )
        )


        # I. CONTEXT RECONSTRUCTION

        if (
            self.context_continuous_head
            is not None
        ):

            context_continuous_prediction = (
                self.context_continuous_head(
                    z_context
                )
            )

        else:

            context_continuous_prediction = (
                z_context.new_empty(
                    (
                        batch_size,
                        0
                    )
                )
            )


        context_categorical_logits = [

            head(
                z_context
            )

            for head
            in self.context_categorical_heads
        ]


        # J. NUISANCE ROUTING

        booklet_logits = (
            self.booklet_classifier(

                grad_reverse(
                    z_general,
                    lambda_booklet_grl
                )
            )
        )


        design_signature_logits = (
            self.design_signature_classifier(

                grad_reverse(
                    z_general,
                    lambda_design_grl
                )
            )
        )


        context_country_logits = (
            self.context_country_classifier(
                z_context
            )
        )


        creative_country_logits = (
            self.creative_country_classifier(

                grad_reverse(
                    z_general,
                    lambda_country_grl
                )
            )
        )


        return {

            "z_general":
                z_general,

            "z_creative":
                z_general,

            "z_context":
                z_context,

            "z_domain_shared":
                z_domain_shared,

            "z_domain_specific":
                z_domain_specific,

            "z_domain_creative":
                z_domain_creative,

            "domain_available":
                domain_available,

            "general_process_tokens":
                general_encoded,

            "general_process_available":
                general_available,

            "credit_logits":
                credit_logits,

            "process_prediction":
                process_prediction,

            "decomposition_credit_logits":
                decomposition_credit_logits,

            "context_continuous_prediction":
                context_continuous_prediction,

            "context_categorical_logits":
                context_categorical_logits,

            "booklet_logits":
                booklet_logits,

            "design_signature_logits":
                design_signature_logits,

            "context_country_logits":
                context_country_logits,

            "creative_country_logits":
                creative_country_logits
        }


print(
    "Final CReST architecture defined."
)

Final CReST architecture defined.


In [ ]:
# Instantiate and smoke-test final CReST architecture

set_global_seed()


model = CReSTFinal(

    item_domain_ids=item_domain_ids,

    item_ideation_ids=item_ideation_ids,

    student_n_continuous=len(
        student_pp.continuous_vars
    ),

    student_cat_cardinalities=(
        student_model_cardinalities
    ),

    school_n_continuous=len(
        school_pp.continuous_vars
    ),

    school_cat_cardinalities=(
        school_model_cardinalities
    ),

    n_train_countries=45,
    n_booklets=30,
    n_design_signatures=10,

    n_domains=4,
    n_processes=3,

    d_model=int(
        CONFIG["d_model"]
    ),

    latent_dim=int(
        CONFIG["latent_dim"]
    ),

    n_heads=int(
        CONFIG["n_heads"]
    ),

    n_layers=int(
        CONFIG["n_layers"]
    ),

    dropout=float(
        CONFIG["dropout"]
    )
).to(device)


batch = next(
    iter(
        train_loader
    )
)


batch = {
    key:
        value[:64]
        .to(device)

    for key, value
    in batch.items()
}


model.eval()


with torch.no_grad():

    output = model(

        batch,

        lambda_booklet_grl=0.0,

        lambda_design_grl=0.0,

        lambda_country_grl=0.0
    )


print(
    "Trainable parameters:",
    sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )
)


print(
    "\nOutput shapes:"
)


for key in [

    "z_general",

    "z_context",

    "z_domain_shared",

    "z_domain_specific",

    "z_domain_creative",

    "general_process_tokens",

    "general_process_available",

    "credit_logits",

    "process_prediction",

    "decomposition_credit_logits",

    "context_continuous_prediction",

    "booklet_logits",

    "design_signature_logits",

    "context_country_logits",

    "creative_country_logits"

]:

    print(
        f"{key:31s}",
        tuple(
            output[
                key
            ].shape
        )
    )


assert (
    output[
        "z_general"
    ].shape
    == (64, 64)
)

assert (
    output[
        "general_process_tokens"
    ].shape
    == (64, 3, 128)
)

assert (
    output[
        "design_signature_logits"
    ].shape
    == (64, 10)
)


print(
    "\nFinal CReST forward smoke test: PASS"
)

Trainable parameters: 1577398

Output shapes:
z_general                       (64, 64)
z_context                       (64, 64)
z_domain_shared                 (64, 4, 64)
z_domain_specific               (64, 4, 64)
z_domain_creative               (64, 4, 64)
general_process_tokens          (64, 3, 128)
general_process_available       (64, 3)
credit_logits                   (64, 32, 3)
process_prediction              (64, 32, 3)
decomposition_credit_logits     (64, 32, 3)
context_continuous_prediction   (64, 2)
booklet_logits                  (64, 30)
design_signature_logits         (64, 10)
context_country_logits          (64, 45)
creative_country_logits         (64, 45)

Final CReST forward smoke test: PASS


In [ ]:
# Final SSL corruption functions

def ensure_one_mask_per_eligible_row(
    selected_mask,
    eligible_mask
):

    needs_one = (
        eligible_mask.any(dim=1)
        &
        ~selected_mask.any(dim=1)
    )


    for row in torch.where(
        needs_one
    )[0]:

        positions = torch.where(
            eligible_mask[row]
        )[0]

        choice = positions[
            torch.randint(
                len(positions),
                (1,),
                device=positions.device
            )
        ]

        selected_mask[
            row,
            choice
        ] = True


    return selected_mask


# Credit + process SSL corruption

def make_main_ssl_corruption(
    batch,
    credit_mask_prob=0.15,
    process_mask_prob=0.15
):

    corrupted = {
        key: value.clone()
        for key, value
        in batch.items()
    }


    # CREDIT

    credit_eligible = (
        batch[
            "performance_mask"
        ]
    )


    credit_ssl_mask = (

        credit_eligible

        &

        (
            torch.rand(
                credit_eligible.shape,
                device=credit_eligible.device
            )
            <
            credit_mask_prob
        )
    )


    credit_ssl_mask = (
        ensure_one_mask_per_eligible_row(

            credit_ssl_mask,

            credit_eligible
        )
    )


    # Original:
    # token 1 -> class 0
    # token 2 -> class 1
    # token 3 -> class 2
    credit_targets = (
        batch[
            "credit_tokens"
        ]
        - 1
    )


    # Dedicated MASK token
    corrupted[
        "credit_tokens"
    ][
        credit_ssl_mask
    ] = 4


    # PROCESS

    process_item_eligible = (
        batch[
            "process_mask"
        ].any(
            dim=-1
        )
    )


    process_ssl_item_mask = (

        process_item_eligible

        &

        (
            torch.rand(
                process_item_eligible.shape,
                device=process_item_eligible.device
            )
            <
            process_mask_prob
        )
    )


    process_ssl_item_mask = (
        ensure_one_mask_per_eligible_row(

            process_ssl_item_mask,

            process_item_eligible
        )
    )


    process_targets = (
        batch[
            "process_values"
        ].clone()
    )


    expanded = (
        process_ssl_item_mask
        .unsqueeze(-1)
        .expand_as(
            batch[
                "process_values"
            ]
        )
    )


    process_reconstruction_mask = (

        expanded

        &

        batch[
            "process_mask"
        ]
    )


    # Main process encoder
    corrupted[
        "process_values"
    ][
        expanded
    ] = 0.0


    # General calibrated branch
    #
    # process_raw is also explicitly cleared.
    corrupted[
        "process_raw"
    ][
        expanded
    ] = 0.0


    # Availability removed from BOTH branches
    corrupted[
        "process_mask"
    ][
        expanded
    ] = False


    return {

        "corrupted_batch":
            corrupted,

        "credit_targets":
            credit_targets,

        "credit_ssl_mask":
            credit_ssl_mask,

        "process_targets":
            process_targets,

        "process_reconstruction_mask":
            process_reconstruction_mask,

        "process_ssl_item_mask":
            process_ssl_item_mask
    }


# Context SSL corruption

def make_context_ssl_corruption(
    batch,
    mask_prob=0.20
):

    corrupted = {
        key: value.clone()
        for key, value
        in batch.items()
    }


    continuous_targets = []
    continuous_masks = []

    categorical_targets = []
    categorical_masks = []


    # Continuous

    for prefix in [
        "student",
        "school"
    ]:

        values = batch[
            f"{prefix}_continuous"
        ]

        observed = batch[
            f"{prefix}_continuous_mask"
        ]


        selected = (

            observed

            &

            (
                torch.rand(
                    observed.shape,
                    device=observed.device
                )
                <
                mask_prob
            )
        )


        continuous_targets.append(
            values.clone()
        )

        continuous_masks.append(
            selected
        )


        corrupted[
            f"{prefix}_continuous"
        ][
            selected
        ] = 0.0


        corrupted[
            f"{prefix}_continuous_mask"
        ][
            selected
        ] = False


    # Categorical

    categorical_specs = [

        (
            "student",
            student_context_mask_ids
        ),

        (
            "school",
            school_context_mask_ids
        )
    ]


    for prefix, mask_ids in (
        categorical_specs
    ):

        values = batch[
            f"{prefix}_categorical"
        ]


        # 0 = missing
        # 1 = unseen
        # >=2 = genuine observed category
        observed = (
            values >= 2
        )


        selected = (

            observed

            &

            (
                torch.rand(
                    observed.shape,
                    device=observed.device
                )
                <
                mask_prob
            )
        )


        categorical_targets.append(
            values.clone()
        )

        categorical_masks.append(
            selected
        )


        for j, mask_id in enumerate(
            mask_ids
        ):

            feature_selected = (
                selected[:, j]
            )


            corrupted[
                f"{prefix}_categorical"
            ][
                feature_selected,
                j
            ] = int(
                mask_id
            )


    return {

        "corrupted_batch":
            corrupted,

        "continuous_targets":
            torch.cat(
                continuous_targets,
                dim=1
            ),

        "continuous_ssl_mask":
            torch.cat(
                continuous_masks,
                dim=1
            ),

        "categorical_targets":
            torch.cat(
                categorical_targets,
                dim=1
            ),

        "categorical_ssl_mask":
            torch.cat(
                categorical_masks,
                dim=1
            )
    }


print(
    "Final SSL corruption functions ready."
)

Final SSL corruption functions ready.


In [ ]:
# Final CReST objective functions

LOSS_WEIGHTS = {

    "credit":
        1.00,

    "process":
        0.50,

    "decomposition_credit":
        0.25,

    "context":
        0.50,

    "decorrelation":
        0.10,

    "booklet":
        0.10,

    "design_signature":
        0.10,

    "country_context":
        0.10,

    "country_creative":
        0.10,

    "global_variance":
        0.05,

    "domain_alignment":
        0.05,

    "domain_orthogonality":
        0.05,

    "domain_variance":
        0.02
}


# Reconstruction losses

def masked_credit_loss(
    logits,
    targets,
    mask
):

    if not mask.any():

        return (
            logits.sum()
            * 0.0
        )


    return F.cross_entropy(
        logits[mask],
        targets[mask]
    )


def masked_process_loss(
    prediction,
    targets,
    mask
):

    if not mask.any():

        return (
            prediction.sum()
            * 0.0
        )


    return F.smooth_l1_loss(

        prediction[mask],

        targets[mask],

        beta=1.0
    )


def context_continuous_loss(
    prediction,
    targets,
    mask
):

    if not mask.any():

        return (
            prediction.sum()
            * 0.0
        )


    return F.smooth_l1_loss(

        prediction[mask],

        targets[mask],

        beta=1.0
    )


def context_categorical_loss(
    logits_list,
    targets,
    mask
):

    losses = []


    for j, logits in enumerate(
        logits_list
    ):

        selected = (
            mask[:, j]
        )


        if selected.any():

            losses.append(

                F.cross_entropy(

                    logits[
                        selected
                    ],

                    targets[
                        selected,
                        j
                    ]
                )
            )


    if not losses:

        return (
            logits_list[0].sum()
            * 0.0
        )


    return torch.stack(
        losses
    ).mean()


# Latent regularisation

def cross_latent_decorrelation(
    z_general,
    z_context,
    eps=1e-6
):

    zg = (
        z_general
        -
        z_general.mean(
            dim=0,
            keepdim=True
        )
    )


    zc = (
        z_context
        -
        z_context.mean(
            dim=0,
            keepdim=True
        )
    )


    zg = zg / (

        zg.std(
            dim=0,
            unbiased=False,
            keepdim=True
        )

        + eps
    )


    zc = zc / (

        zc.std(
            dim=0,
            unbiased=False,
            keepdim=True
        )

        + eps
    )


    correlation = (

        zg.T
        @
        zc

    ) / zg.shape[0]


    return (
        correlation
        .pow(2)
        .mean()
    )


def variance_loss(
    z,
    target_std=1.0
):

    std = torch.sqrt(

        z.var(
            dim=0,
            unbiased=False
        )

        + 1e-4
    )


    return F.relu(
        target_std - std
    ).mean()


def domain_alignment_loss(
    z_general,
    z_shared,
    available
):

    if not available.any():

        return (
            z_shared.sum()
            * 0.0
        )


    general = (
        z_general
        .unsqueeze(1)
        .expand_as(
            z_shared
        )
    )


    cosine = F.cosine_similarity(

        general[
            available
        ],

        z_shared[
            available
        ],

        dim=-1
    )


    return (
        1.0
        -
        cosine
    ).mean()


def domain_orthogonality_loss(
    z_general,
    z_specific,
    available
):

    if not available.any():

        return (
            z_specific.sum()
            * 0.0
        )


    general = (
        z_general
        .unsqueeze(1)
        .expand_as(
            z_specific
        )
    )


    cosine = F.cosine_similarity(

        general[
            available
        ],

        z_specific[
            available
        ],

        dim=-1
    )


    return (
        cosine
        .pow(2)
        .mean()
    )


def domain_variance_loss(
    z_specific,
    available,
    target_std=0.5
):

    losses = []


    for domain_id in range(
        z_specific.shape[1]
    ):

        valid = (
            available[
                :,
                domain_id
            ]
        )


        if valid.sum() < 2:
            continue


        std = torch.sqrt(

            z_specific[
                valid,
                domain_id
            ].var(
                dim=0,
                unbiased=False
            )

            + 1e-4
        )


        losses.append(

            F.relu(
                target_std
                -
                std
            ).mean()
        )


    if not losses:

        return (
            z_specific.sum()
            * 0.0
        )


    return torch.stack(
        losses
    ).mean()


# Final composite objective

def compute_final_objective(
    model,
    batch,

    lambda_booklet_grl=1.0,
    lambda_design_grl=1.0,
    lambda_country_grl=1.0,

    credit_mask_prob=0.15,
    process_mask_prob=0.15,
    context_mask_prob=0.20
):

    # SSL corruption

    main_ssl = (
        make_main_ssl_corruption(

            batch,

            credit_mask_prob=(
                credit_mask_prob
            ),

            process_mask_prob=(
                process_mask_prob
            )
        )
    )


    context_ssl = (
        make_context_ssl_corruption(

            main_ssl[
                "corrupted_batch"
            ],

            mask_prob=(
                context_mask_prob
            )
        )
    )


    corrupted_batch = (
        context_ssl[
            "corrupted_batch"
        ]
    )


    # Forward

    output = model(

        corrupted_batch,

        lambda_booklet_grl=(
            lambda_booklet_grl
        ),

        lambda_design_grl=(
            lambda_design_grl
        ),

        lambda_country_grl=(
            lambda_country_grl
        )
    )


    # Credit reconstruction

    loss_credit = masked_credit_loss(

        output[
            "credit_logits"
        ],

        main_ssl[
            "credit_targets"
        ],

        main_ssl[
            "credit_ssl_mask"
        ]
    )


    # Process reconstruction

    loss_process = masked_process_loss(

        output[
            "process_prediction"
        ],

        main_ssl[
            "process_targets"
        ],

        main_ssl[
            "process_reconstruction_mask"
        ]
    )


    # Domain decomposition reconstruction

    loss_decomposition = (
        masked_credit_loss(

            output[
                "decomposition_credit_logits"
            ],

            main_ssl[
                "credit_targets"
            ],

            main_ssl[
                "credit_ssl_mask"
            ]
        )
    )


    # Context reconstruction

    loss_context_cont = (
        context_continuous_loss(

            output[
                "context_continuous_prediction"
            ],

            context_ssl[
                "continuous_targets"
            ],

            context_ssl[
                "continuous_ssl_mask"
            ]
        )
    )


    loss_context_cat = (
        context_categorical_loss(

            output[
                "context_categorical_logits"
            ],

            context_ssl[
                "categorical_targets"
            ],

            context_ssl[
                "categorical_ssl_mask"
            ]
        )
    )


    loss_context = (
        0.5
        *
        (
            loss_context_cont
            +
            loss_context_cat
        )
    )


    # Representation regularisation

    loss_decorrelation = (
        cross_latent_decorrelation(

            output[
                "z_general"
            ],

            output[
                "z_context"
            ]
        )
    )


    loss_global_variance = (
        variance_loss(
            output[
                "z_general"
            ]
        )
    )


    loss_domain_alignment = (
        domain_alignment_loss(

            output[
                "z_general"
            ],

            output[
                "z_domain_shared"
            ],

            output[
                "domain_available"
            ]
        )
    )


    loss_domain_orthogonality = (
        domain_orthogonality_loss(

            output[
                "z_general"
            ],

            output[
                "z_domain_specific"
            ],

            output[
                "domain_available"
            ]
        )
    )


    loss_domain_variance = (
        domain_variance_loss(

            output[
                "z_domain_specific"
            ],

            output[
                "domain_available"
            ]
        )
    )


    # Nuisance routing

    loss_booklet = F.cross_entropy(

        output[
            "booklet_logits"
        ],

        batch[
            "booklet_nuisance_id"
        ]
    )


    loss_design = F.cross_entropy(

        output[
            "design_signature_logits"
        ],

        batch[
            "design_signature_id"
        ]
    )


    country_targets = (
        batch[
            "country_train_id"
        ]
    )


    loss_country_context = (
        F.cross_entropy(

            output[
                "context_country_logits"
            ],

            country_targets
        )
    )


    loss_country_creative = (
        F.cross_entropy(

            output[
                "creative_country_logits"
            ],

            country_targets
        )
    )


    # Collect

    losses = {

        "credit":
            loss_credit,

        "process":
            loss_process,

        "decomposition_credit":
            loss_decomposition,

        "context":
            loss_context,

        "context_continuous":
            loss_context_cont,

        "context_categorical":
            loss_context_cat,

        "decorrelation":
            loss_decorrelation,

        "booklet":
            loss_booklet,

        "design_signature":
            loss_design,

        "country_context":
            loss_country_context,

        "country_creative":
            loss_country_creative,

        "global_variance":
            loss_global_variance,

        "domain_alignment":
            loss_domain_alignment,

        "domain_orthogonality":
            loss_domain_orthogonality,

        "domain_variance":
            loss_domain_variance
    }


    total_loss = sum(

        LOSS_WEIGHTS[name]
        *
        losses[name]

        for name
        in LOSS_WEIGHTS
    )


    return (
        total_loss,
        losses,
        output,
        main_ssl
    )


print(
    "Final CReST objective ready."
)

Final CReST objective ready.


In [ ]:
# Final backward and leakage-safety smoke test

SMOKE_N = 64


smoke_batch = next(
    iter(
        train_loader
    )
)


smoke_batch = {

    key:
        value[:SMOKE_N]
        .to(device)

    for key, value
    in smoke_batch.items()
}


set_global_seed(
    GLOBAL_SEED + 1000
)


model.train()

model.zero_grad(
    set_to_none=True
)


total_loss, losses, output, ssl_info = (
    compute_final_objective(

        model,

        smoke_batch,

        lambda_booklet_grl=0.5,

        lambda_design_grl=0.5,

        lambda_country_grl=0.5
    )
)


assert torch.isfinite(
    total_loss
)


total_loss.backward()


# Gradient helper

def grad_norm(
    parameter
):

    if parameter.grad is None:
        return float("nan")

    return float(
        parameter.grad
        .detach()
        .norm()
        .cpu()
    )


critical_gradients = {

    "general_token_projection":
        grad_norm(
            model
            .general_token_projection[0]
            .weight
        ),

    "general_process_encoder":
        grad_norm(
            model
            .general_process_encoder
            .layers[0]
            .self_attn
            .in_proj_weight
        ),

    "creative_projection":
        grad_norm(
            model
            .creative_projection[0]
            .weight
        ),

    "item_embedding_decoder_only":
        grad_norm(
            model
            .item_embedding
            .weight
        ),

    "design_signature_classifier":
        grad_norm(
            model
            .design_signature_classifier[0]
            .weight
        ),

    "booklet_classifier":
        grad_norm(
            model
            .booklet_classifier[0]
            .weight
        ),

    "creative_country_classifier":
        grad_norm(
            model
            .creative_country_classifier[0]
            .weight
        )
}


# SSL safety checks

assert (
    ssl_info[
        "credit_ssl_mask"
    ].any()
)


assert (
    ssl_info[
        "process_reconstruction_mask"
    ].any()
)


# Report

print(
    "Total loss:",
    round(
        float(
            total_loss.detach().cpu()
        ),
        6
    )
)


print(
    "\nLoss components:"
)


for name, value in (
    losses.items()
):

    print(
        f"{name:26s}",
        f"{float(value.detach().cpu()):.6f}"
    )


print(
    "\nCritical gradient norms:"
)


for name, value in (
    critical_gradients.items()
):

    print(
        f"{name:31s}",
        f"{value:.6f}"
    )


assert all(

    np.isfinite(value)
    and value > 0

    for value
    in critical_gradients.values()
)


print(
    "\nMasked credit cells:",
    int(
        ssl_info[
            "credit_ssl_mask"
        ].sum()
    )
)


print(
    "Masked process values:",
    int(
        ssl_info[
            "process_reconstruction_mask"
        ].sum()
    )
)


print(
    "\nFinal CReST backward smoke test: PASS"
)

Total loss: 4.220828

Loss components:
credit                     1.423139
process                    0.746464
decomposition_credit       1.563097
context                    1.220312
context_continuous         0.512220
context_categorical        1.928403
decorrelation              0.015509
booklet                    3.391360
design_signature           2.312465
country_context            3.826960
country_creative           3.918859
global_variance            0.480983
domain_alignment           0.970935
domain_orthogonality       0.029483
domain_variance            0.147051

Critical gradient norms:
general_token_projection        0.109031
general_process_encoder         0.202383
creative_projection             0.723565
item_embedding_decoder_only     0.073453
design_signature_classifier     0.034307
booklet_classifier              0.056609
creative_country_classifier     0.047887

Masked credit cells: 116
Masked process values: 354

Final CReST backward smoke test: PASS


In [ ]:
# Final pilot validation utilities

FINAL_PILOT_EPOCHS = 5


def grl_schedule(
    progress
):

    return (
        2.0
        /
        (
            1.0
            +
            np.exp(
                -10.0
                * progress
            )
        )
        -
        1.0
    )


def reconstruction_score_from_losses(
    losses
):

    return (

        losses["credit"]

        +

        LOSS_WEIGHTS[
            "process"
        ]
        *
        losses["process"]

        +

        LOSS_WEIGHTS[
            "decomposition_credit"
        ]
        *
        losses[
            "decomposition_credit"
        ]

        +

        LOSS_WEIGHTS[
            "context"
        ]
        *
        losses["context"]
    )


@torch.no_grad()
def evaluate_final_country_equal_reconstruction(
    model,
    seed=GLOBAL_SEED + 1500
):

    model.eval()


    validation_countries = (
        fold0[
            "validation"
        ]["CNT"]
    )


    country_scores = {}


    # Fixed SSL corruption across epochs
    with torch.random.fork_rng():

        torch.manual_seed(
            seed
        )


        for country in sorted(
            np.unique(
                validation_countries
            )
        ):

            local_indices = np.where(
                validation_countries
                == country
            )[0]


            subset = torch.utils.data.Subset(
                validation_dataset,
                local_indices.tolist()
            )


            loader = DataLoader(

                subset,

                batch_size=BATCH_SIZE,

                shuffle=False,

                num_workers=0
            )


            total_score = 0.0
            total_students = 0


            for batch in loader:

                batch = {
                    key:
                        value.to(device)

                    for key, value
                    in batch.items()
                }


                main_ssl = (
                    make_main_ssl_corruption(

                        batch,

                        credit_mask_prob=0.15,

                        process_mask_prob=0.15
                    )
                )


                context_ssl = (
                    make_context_ssl_corruption(

                        main_ssl[
                            "corrupted_batch"
                        ],

                        mask_prob=0.20
                    )
                )


                out = model(

                    context_ssl[
                        "corrupted_batch"
                    ],

                    lambda_booklet_grl=0.0,

                    lambda_design_grl=0.0,

                    lambda_country_grl=0.0
                )


                val_losses = {}


                val_losses["credit"] = (
                    masked_credit_loss(

                        out[
                            "credit_logits"
                        ],

                        main_ssl[
                            "credit_targets"
                        ],

                        main_ssl[
                            "credit_ssl_mask"
                        ]
                    )
                )


                val_losses["process"] = (
                    masked_process_loss(

                        out[
                            "process_prediction"
                        ],

                        main_ssl[
                            "process_targets"
                        ],

                        main_ssl[
                            "process_reconstruction_mask"
                        ]
                    )
                )


                val_losses[
                    "decomposition_credit"
                ] = masked_credit_loss(

                    out[
                        "decomposition_credit_logits"
                    ],

                    main_ssl[
                        "credit_targets"
                    ],

                    main_ssl[
                        "credit_ssl_mask"
                    ]
                )


                context_cont = (
                    context_continuous_loss(

                        out[
                            "context_continuous_prediction"
                        ],

                        context_ssl[
                            "continuous_targets"
                        ],

                        context_ssl[
                            "continuous_ssl_mask"
                        ]
                    )
                )


                context_cat = (
                    context_categorical_loss(

                        out[
                            "context_categorical_logits"
                        ],

                        context_ssl[
                            "categorical_targets"
                        ],

                        context_ssl[
                            "categorical_ssl_mask"
                        ]
                    )
                )


                val_losses["context"] = (
                    0.5
                    *
                    (
                        context_cont
                        +
                        context_cat
                    )
                )


                score = (
                    reconstruction_score_from_losses(
                        val_losses
                    )
                )


                n_batch = (
                    batch[
                        "credit_tokens"
                    ].shape[0]
                )


                total_score += (
                    float(
                        score.cpu()
                    )
                    *
                    n_batch
                )


                total_students += (
                    n_batch
                )


            country_scores[
                country
            ] = (
                total_score
                /
                total_students
            )


    country_equal_score = float(
        np.mean(
            list(
                country_scores.values()
            )
        )
    )


    return (
        country_equal_score,
        country_scores
    )


print(
    "Final pilot utilities ready."
)

Final pilot utilities ready.


In [ ]:
# Final CReST Fold-0 architecture pilot

# Fresh sampler

final_sampler = WeightedRandomSampler(

    weights=torch.as_tensor(
        sampling_weights,
        dtype=torch.double
    ),

    num_samples=len(
        train_dataset
    ),

    replacement=True,

    generator=(
        torch.Generator()
        .manual_seed(
            GLOBAL_SEED + 1700
        )
    )
)


final_train_loader = DataLoader(

    train_dataset,

    batch_size=BATCH_SIZE,

    sampler=final_sampler,

    shuffle=False,

    num_workers=0,

    pin_memory=torch.cuda.is_available()
)


# Fresh model

set_global_seed(
    GLOBAL_SEED
)


model = CReSTFinal(

    item_domain_ids=item_domain_ids,

    item_ideation_ids=item_ideation_ids,

    student_n_continuous=len(
        student_pp.continuous_vars
    ),

    student_cat_cardinalities=(
        student_model_cardinalities
    ),

    school_n_continuous=len(
        school_pp.continuous_vars
    ),

    school_cat_cardinalities=(
        school_model_cardinalities
    ),

    n_train_countries=45,

    n_booklets=30,

    n_design_signatures=10,

    n_domains=4,

    n_processes=3,

    d_model=int(
        CONFIG["d_model"]
    ),

    latent_dim=int(
        CONFIG["latent_dim"]
    ),

    n_heads=int(
        CONFIG["n_heads"]
    ),

    n_layers=int(
        CONFIG["n_layers"]
    ),

    dropout=float(
        CONFIG["dropout"]
    )
).to(device)


# Optimisation

optimizer = torch.optim.AdamW(

    model.parameters(),

    lr=1e-4,

    weight_decay=1e-4
)


scheduler = (
    torch.optim.lr_scheduler.ReduceLROnPlateau(

        optimizer,

        mode="min",

        factor=0.5,

        patience=2,

        min_lr=1e-6
    )
)


# Checkpoint

final_checkpoint = (
    CHECKPOINT_DIR
    / "crest_final_fold0_pilot_best.pt"
)


history = []

best_val = np.inf
best_epoch = None


# Training

for epoch in range(
    1,
    FINAL_PILOT_EPOCHS + 1
):

    model.train()


    running_total = 0.0
    running_recon = 0.0

    running_booklet = 0.0
    running_design = 0.0

    running_country_context = 0.0
    running_country_creative = 0.0

    n_seen = 0


    for batch_idx, batch in enumerate(
        final_train_loader
    ):

        batch = {
            key:
                value.to(device)

            for key, value
            in batch.items()
        }


        # GRL schedule

        progress = (

            (
                epoch - 1
            )

            +

            batch_idx
            /
            max(
                len(
                    final_train_loader
                ),
                1
            )

        ) / FINAL_PILOT_EPOCHS


        lambda_grl = float(
            grl_schedule(
                progress
            )
        )


        # Forward / backward

        optimizer.zero_grad(
            set_to_none=True
        )


        total_loss, losses, _, _ = (
            compute_final_objective(

                model,

                batch,

                lambda_booklet_grl=(
                    lambda_grl
                ),

                lambda_design_grl=(
                    lambda_grl
                ),

                lambda_country_grl=(
                    lambda_grl
                )
            )
        )


        total_loss.backward()


        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )


        optimizer.step()


        # Running metrics

        n_batch = (
            batch[
                "credit_tokens"
            ].shape[0]
        )


        recon = (
            reconstruction_score_from_losses(
                losses
            )
        )


        running_total += (
            float(
                total_loss.detach().cpu()
            )
            *
            n_batch
        )


        running_recon += (
            float(
                recon.detach().cpu()
            )
            *
            n_batch
        )


        running_booklet += (
            float(
                losses[
                    "booklet"
                ].detach().cpu()
            )
            *
            n_batch
        )


        running_design += (
            float(
                losses[
                    "design_signature"
                ].detach().cpu()
            )
            *
            n_batch
        )


        running_country_context += (
            float(
                losses[
                    "country_context"
                ].detach().cpu()
            )
            *
            n_batch
        )


        running_country_creative += (
            float(
                losses[
                    "country_creative"
                ].detach().cpu()
            )
            *
            n_batch
        )


        n_seen += (
            n_batch
        )


    # Unseen-country validation

    val_score, val_country_scores = (
        evaluate_final_country_equal_reconstruction(
            model
        )
    )


    scheduler.step(
        val_score
    )


    lr_now = (
        optimizer
        .param_groups[0]["lr"]
    )


    epoch_row = {

        "epoch":
            epoch,

        "train_total":
            running_total
            / n_seen,

        "train_reconstruction":
            running_recon
            / n_seen,

        "train_booklet_ce":
            running_booklet
            / n_seen,

        "train_design_signature_ce":
            running_design
            / n_seen,

        "train_country_context_ce":
            running_country_context
            / n_seen,

        "train_country_creative_ce":
            running_country_creative
            / n_seen,

        "val_country_equal_reconstruction":
            val_score,

        "learning_rate":
            lr_now
    }


    history.append(
        epoch_row
    )


    # Best checkpoint

    if val_score < best_val:

        best_val = (
            val_score
        )

        best_epoch = (
            epoch
        )


        torch.save(

            {
                "epoch":
                    epoch,

                "model_state_dict":
                    model.state_dict(),

                "optimizer_state_dict":
                    optimizer.state_dict(),

                "val_country_equal_reconstruction":
                    val_score,

                "val_country_scores":
                    val_country_scores
            },

            final_checkpoint
        )


    print(

        f"Epoch {epoch:02d} | "

        f"train={epoch_row['train_total']:.4f} | "

        f"recon={epoch_row['train_reconstruction']:.4f} | "

        f"val={val_score:.4f} | "

        f"bookletCE={epoch_row['train_booklet_ce']:.4f} | "

        f"designCE={epoch_row['train_design_signature_ce']:.4f} | "

        f"ctxCountryCE={epoch_row['train_country_context_ce']:.4f} | "

        f"creativeCountryCE={epoch_row['train_country_creative_ce']:.4f} | "

        f"GRL={lambda_grl:.3f} | "

        f"lr={lr_now:.1e}"
    )


# Save pilot history

history_df = pd.DataFrame(
    history
)


history_df.to_csv(

    RESULTS_DIR
    / "fold0_final_architecture_pilot_history.csv",

    index=False
)


print(
    "\nBest final pilot epoch:",
    best_epoch
)

print(
    "Best country-equal validation reconstruction:",
    round(
        best_val,
        6
    )
)

print(
    "Checkpoint:",
    final_checkpoint
)

Epoch 01 | train=2.9853 | recon=1.6521 | val=1.4833 | bookletCE=3.2672 | designCE=2.3060 | ctxCountryCE=3.6838 | creativeCountryCE=3.7501 | GRL=0.761 | lr=1.0e-04
Epoch 02 | train=2.7756 | recon=1.4960 | val=1.4439 | bookletCE=3.1701 | designCE=2.3089 | ctxCountryCE=3.2438 | creativeCountryCE=3.8021 | GRL=0.964 | lr=1.0e-04
Epoch 03 | train=2.7044 | recon=1.4664 | val=1.4437 | bookletCE=3.0370 | designCE=2.3083 | ctxCountryCE=2.9686 | creativeCountryCE=3.7986 | GRL=0.995 | lr=1.0e-04
Epoch 04 | train=2.6664 | recon=1.4584 | val=1.4239 | bookletCE=2.8965 | designCE=2.3055 | ctxCountryCE=2.8477 | creativeCountryCE=3.8136 | GRL=0.999 | lr=1.0e-04
Epoch 05 | train=2.6322 | recon=1.4455 | val=1.4234 | bookletCE=2.8234 | designCE=2.3021 | ctxCountryCE=2.7781 | creativeCountryCE=3.8110 | GRL=1.000 | lr=1.0e-04

Best final pilot epoch: 5
Best country-equal validation reconstruction: 1.42341
Checkpoint: PROJECT_ROOT\checkpoints\fold_0\crest_final_fold0_pilot_best.pt


In [ ]:
# Restore best final checkpoint and extract frozen latents
# Train + Validation only. Test remains untouched.

# Restore final model

set_global_seed(
    GLOBAL_SEED
)


model = CReSTFinal(

    item_domain_ids=item_domain_ids,
    item_ideation_ids=item_ideation_ids,

    student_n_continuous=len(
        student_pp.continuous_vars
    ),

    student_cat_cardinalities=(
        student_model_cardinalities
    ),

    school_n_continuous=len(
        school_pp.continuous_vars
    ),

    school_cat_cardinalities=(
        school_model_cardinalities
    ),

    n_train_countries=45,
    n_booklets=30,
    n_design_signatures=10,

    n_domains=4,
    n_processes=3,

    d_model=int(
        CONFIG["d_model"]
    ),

    latent_dim=int(
        CONFIG["latent_dim"]
    ),

    n_heads=int(
        CONFIG["n_heads"]
    ),

    n_layers=int(
        CONFIG["n_layers"]
    ),

    dropout=float(
        CONFIG["dropout"]
    )
).to(device)


checkpoint = torch.load(
    final_checkpoint,
    map_location=device
)


model.load_state_dict(
    checkpoint[
        "model_state_dict"
    ]
)

model.eval()


print(
    "Restored epoch:",
    checkpoint["epoch"]
)

print(
    "Validation reconstruction:",
    round(
        checkpoint[
            "val_country_equal_reconstruction"
        ],
        6
    )
)


# Deterministic loaders

PROBE_BATCH_SIZE = 512


probe_train_loader = DataLoader(

    train_dataset,

    batch_size=PROBE_BATCH_SIZE,

    shuffle=False,

    num_workers=0
)


probe_validation_loader = DataLoader(

    validation_dataset,

    batch_size=PROBE_BATCH_SIZE,

    shuffle=False,

    num_workers=0
)


# Frozen latent extraction

@torch.no_grad()
def extract_final_latents(
    model,
    loader
):

    z_general_list = []
    z_context_list = []

    booklet_list = []
    design_list = []


    for batch in loader:

        batch = {
            key:
                value.to(device)

            for key, value
            in batch.items()
        }


        output = model(

            batch,

            lambda_booklet_grl=0.0,

            lambda_design_grl=0.0,

            lambda_country_grl=0.0
        )


        z_general_list.append(

            output[
                "z_general"
            ]
            .cpu()
            .numpy()
        )


        z_context_list.append(

            output[
                "z_context"
            ]
            .cpu()
            .numpy()
        )


        booklet_list.append(

            batch[
                "booklet_nuisance_id"
            ]
            .cpu()
            .numpy()
        )


        design_list.append(

            batch[
                "design_signature_id"
            ]
            .cpu()
            .numpy()
        )


    return {

        "z_general":
            np.concatenate(
                z_general_list,
                axis=0
            ),

        "z_context":
            np.concatenate(
                z_context_list,
                axis=0
            ),

        "booklet":
            np.concatenate(
                booklet_list,
                axis=0
            ),

        "design_signature":
            np.concatenate(
                design_list,
                axis=0
            )
    }


final_train_latents = (
    extract_final_latents(
        model,
        probe_train_loader
    )
)


final_validation_latents = (
    extract_final_latents(
        model,
        probe_validation_loader
    )
)


print(
    "\nTrain z_general:",
    final_train_latents[
        "z_general"
    ].shape
)

print(
    "Validation z_general:",
    final_validation_latents[
        "z_general"
    ].shape
)

print(
    "Booklet classes:",
    len(
        np.unique(
            final_validation_latents[
                "booklet"
            ]
        )
    )
)

print(
    "Design-signature classes:",
    len(
        np.unique(
            final_validation_latents[
                "design_signature"
            ]
        )
    )
)

Restored epoch: 5
Validation reconstruction: 1.42341

Train z_general: (99966, 64)
Validation z_general: (20949, 64)
Booklet classes: 30
Design-signature classes: 10


In [ ]:
# Final frozen cross-system nuisance probes

def run_frozen_probe(
    X_train,
    y_train,
    X_validation,
    y_validation,
    latent_name,
    target_name
):

    probe = make_pipeline(

        StandardScaler(),

        LogisticRegression(
            max_iter=500,
            solver="lbfgs",
            class_weight="balanced",
            random_state=GLOBAL_SEED
        )
    )


    probe.fit(
        X_train,
        y_train
    )


    prediction = probe.predict(
        X_validation
    )


    return {

        "latent":
            latent_name,

        "target":
            target_name,

        "accuracy":
            accuracy_score(
                y_validation,
                prediction
            ),

        "balanced_accuracy":
            balanced_accuracy_score(
                y_validation,
                prediction
            ),

        "macro_f1":
            f1_score(
                y_validation,
                prediction,
                average="macro"
            ),

        "chance_BA":
            1.0
            /
            len(
                np.unique(
                    y_validation
                )
            )
    }


probe_results = []


for latent_name in [
    "z_general",
    "z_context"
]:

    # 30-class booklet probe

    probe_results.append(

        run_frozen_probe(

            final_train_latents[
                latent_name
            ],

            final_train_latents[
                "booklet"
            ],

            final_validation_latents[
                latent_name
            ],

            final_validation_latents[
                "booklet"
            ],

            latent_name,

            "booklet_30"
        )
    )


    # 10-class assessment-design signature probe

    probe_results.append(

        run_frozen_probe(

            final_train_latents[
                latent_name
            ],

            final_train_latents[
                "design_signature"
            ],

            final_validation_latents[
                latent_name
            ],

            final_validation_latents[
                "design_signature"
            ],

            latent_name,

            "design_signature_10"
        )
    )


final_probe_df = pd.DataFrame(
    probe_results
)


final_probe_df.to_csv(

    RESULTS_DIR
    / "fold0_final_architecture_nuisance_probes.csv",

    index=False
)


print(
    final_probe_df.to_string(
        index=False
    )
)


print(
    "\nKey references:"
)

print(
    "V3 z_general booklet BA          : 0.3389"
)

print(
    "V5 z_general booklet BA          : 0.3302"
)

print(
    "V6 z_general booklet BA          : 0.1268"
)

print(
    "V6 z_general design-signature BA : 0.3756"
)

print(
    "Calibrated input design BA       : 0.1724"
)

print(
    "Booklet chance BA                : 0.0333"
)

print(
    "Design-signature chance BA       : 0.1000"
)

   latent              target  accuracy  balanced_accuracy  macro_f1  chance_BA
z_general          booklet_30  0.059716           0.091034  0.048264   0.033333
z_general design_signature_10  0.238579           0.239282  0.233365   0.100000
z_context          booklet_30  0.006874           0.032036  0.007850   0.033333
z_context design_signature_10  0.100530           0.100632  0.097327   0.100000

Key references:
V3 z_general booklet BA          : 0.3389
V5 z_general booklet BA          : 0.3302
V6 z_general booklet BA          : 0.1268
V6 z_general design-signature BA : 0.3756
Calibrated input design BA       : 0.1724
Booklet chance BA                : 0.0333
Design-signature chance BA       : 0.1000


In [ ]:
# Disjoint-item creative-signal retention in frozen z_general
# NO neural-network training.

from sklearn.model_selection import train_test_split
from sklearn.linear_model import RidgeCV
from sklearn.metrics import (
    r2_score,
    mean_absolute_error
)
from scipy.stats import spearmanr


# Recreate the same deterministic 16/16 split used in
# Notebook 03

domain_ids_np = np.asarray(
    item_domain_ids,
    dtype=int
)

process_ids_np = np.asarray(
    item_ideation_ids,
    dtype=int
)

all_items = np.arange(32)

selected_split = None


for seed in range(
    GLOBAL_SEED,
    GLOBAL_SEED + 500
):

    item_A, item_B = train_test_split(

        all_items,

        test_size=0.50,

        random_state=seed,

        stratify=process_ids_np
    )


    domain_A = np.bincount(
        domain_ids_np[item_A],
        minlength=4
    )

    domain_B = np.bincount(
        domain_ids_np[item_B],
        minlength=4
    )


    process_A = np.bincount(
        process_ids_np[item_A],
        minlength=3
    )

    process_B = np.bincount(
        process_ids_np[item_B],
        minlength=3
    )


    if (
        np.all(domain_A >= 2)
        and
        np.all(domain_B >= 2)
        and
        np.max(
            np.abs(
                process_A
                -
                process_B
            )
        ) <= 1
    ):

        selected_split = (
            np.sort(item_A),
            np.sort(item_B),
            seed
        )

        break


assert selected_split is not None

item_A, item_B, split_seed = (
    selected_split
)


print(
    "Recovered item-split seed:",
    split_seed
)

print(
    "Set A:",
    item_A.tolist()
)

print(
    "Set B:",
    item_B.tolist()
)


# Mask target items completely before latent extraction

def remove_target_items(
    batch,
    target_items
):

    masked = {
        key: value.clone()
        for key, value
        in batch.items()
    }


    target_items_t = torch.as_tensor(
        target_items,
        dtype=torch.long,
        device=masked[
            "credit_tokens"
        ].device
    )


    # Credit unavailable
    masked[
        "credit_tokens"
    ][
        :,
        target_items_t
    ] = 0


    masked[
        "performance_mask"
    ][
        :,
        target_items_t
    ] = False


    # Behaviour unavailable
    masked[
        "behaviour_tokens"
    ][
        :,
        target_items_t
    ] = 0


    masked[
        "interaction_mask"
    ][
        :,
        target_items_t
    ] = False


    # Process unavailable
    masked[
        "process_values"
    ][
        :,
        target_items_t,
        :
    ] = 0.0


    masked[
        "process_raw"
    ][
        :,
        target_items_t,
        :
    ] = 0.0


    masked[
        "process_mask"
    ][
        :,
        target_items_t,
        :
    ] = False


    return masked


# Extract frozen source-only z_general

@torch.no_grad()
def extract_source_only_latent(
    loader,
    target_items
):

    latents = []


    for batch in loader:

        batch = {
            key:
                value.to(device)

            for key, value
            in batch.items()
        }


        batch = remove_target_items(
            batch,
            target_items
        )


        output = model(

            batch,

            lambda_booklet_grl=0.0,

            lambda_design_grl=0.0,

            lambda_country_grl=0.0
        )


        latents.append(

            output[
                "z_general"
            ]
            .cpu()
            .numpy()
        )


    return np.concatenate(
        latents,
        axis=0
    )


# Held-out target score

def heldout_credit_target(
    split_data,
    target_items,
    min_scorable_items=3
):

    tokens = split_data[
        "credit_tokens"
    ][
        :,
        target_items
    ]


    mask = split_data[
        "performance_mask"
    ][
        :,
        target_items
    ]


    scores = (

        tokens.astype(
            np.float32
        )
        - 1.0

    ) / 2.0


    n_scorable = (
        mask.sum(
            axis=1
        )
    )


    target = np.divide(

        (
            scores
            * mask
        ).sum(
            axis=1
        ),

        n_scorable,

        out=np.full(
            len(tokens),
            np.nan,
            dtype=np.float32
        ),

        where=(
            n_scorable > 0
        )
    )


    eligible = (
        n_scorable
        >= min_scorable_items
    )


    return (
        target,
        eligible
    )


# Evaluate A -> B and B -> A

signal_rows = []


for source_name, source_items, target_items in [

    (
        "A_to_B",
        item_A,
        item_B
    ),

    (
        "B_to_A",
        item_B,
        item_A
    )
]:

    print(
        f"\nExtracting latent for {source_name}..."
    )


    # target_items are completely hidden;
    # therefore latent is based only on source_items.
    Z_train = (
        extract_source_only_latent(
            probe_train_loader,
            target_items
        )
    )


    Z_validation = (
        extract_source_only_latent(
            probe_validation_loader,
            target_items
        )
    )


    y_train, eligible_train = (
        heldout_credit_target(
            fold0["train"],
            target_items
        )
    )


    y_validation, eligible_validation = (
        heldout_credit_target(
            fold0["validation"],
            target_items
        )
    )


    probe = make_pipeline(

        StandardScaler(),

        RidgeCV(
            alphas=[
                0.1,
                1.0,
                10.0,
                100.0
            ]
        )
    )


    probe.fit(

        Z_train[
            eligible_train
        ],

        y_train[
            eligible_train
        ]
    )


    prediction = probe.predict(

        Z_validation[
            eligible_validation
        ]
    )


    y_true = (
        y_validation[
            eligible_validation
        ]
    )


    signal_rows.append(
        {
            "direction":
                source_name,

            "n_train":
                int(
                    eligible_train.sum()
                ),

            "n_validation":
                int(
                    eligible_validation.sum()
                ),

            "R2":
                r2_score(
                    y_true,
                    prediction
                ),

            "MAE":
                mean_absolute_error(
                    y_true,
                    prediction
                ),

            "Spearman_rho":
                spearmanr(
                    y_true,
                    prediction
                ).statistic
        }
    )


latent_signal_df = pd.DataFrame(
    signal_rows
)


print(
    "\nFrozen z_general disjoint-item results:"
)

print(
    latent_signal_df.to_string(
        index=False
    )
)


print(
    "\nMean:"
)

print(
    latent_signal_df[
        [
            "R2",
            "MAE",
            "Spearman_rho"
        ]
    ]
    .mean()
    .to_string()
)


latent_signal_df.to_csv(

    RESULTS_DIR
    / "fold0_final_zgeneral_disjoint_item_signal.csv",

    index=False
)

Recovered item-split seed: 20261005
Set A: [0, 1, 5, 6, 7, 8, 9, 10, 12, 15, 17, 20, 24, 27, 28, 31]
Set B: [2, 3, 4, 11, 13, 14, 16, 18, 19, 21, 22, 23, 25, 26, 29, 30]

Extracting latent for A_to_B...

Extracting latent for B_to_A...

Frozen z_general disjoint-item results:
direction  n_train  n_validation       R2      MAE  Spearman_rho
   A_to_B    94046         19312 0.388312 0.172597      0.643714
   B_to_A    95332         19370 0.431026 0.159581      0.668226

Mean:
R2              0.409669
MAE             0.166089
Spearman_rho    0.655970


In [ ]:
# Lock the final CReST architecture

ARCHITECTURE_LOCK = {

    "architecture_name":
        "CReST-Final",

    "architecture_status":
        "LOCKED",

    "fold0_validation_reconstruction":
        1.423410,

    "booklet_probe_BA": {

        "V3":
            0.3389,

        "V5":
            0.330203,

        "V6":
            0.126755,

        "final":
            0.091034,

        "chance":
            1.0 / 30.0
    },

    "design_signature_probe_BA": {

        "V6":
            0.375590,

        "final":
            0.239282,

        "chance":
            0.100000
    },

    "disjoint_item_latent_signal": {

        "A_to_B_R2":
            0.388312,

        "A_to_B_Spearman":
            0.643714,

        "B_to_A_R2":
            0.431026,

        "B_to_A_Spearman":
            0.668226,

        "mean_R2":
            0.409669,

        "mean_Spearman":
            0.655970
    },

    "general_representation":
        (
            "train-only item-conditioned residualization -> "
            "country-equal assessment-design calibration -> "
            "GDI/GCI/EII calibrated process tokens -> "
            "3-token Transformer -> equal available-process pooling"
        ),

    "interpretation":
        (
            "Assessment-design-robust rather than fully "
            "assessment-design-invariant."
        ),

    "test_status":
        "Fold-0 test set untouched",

    "architecture_search":
        "CLOSED"
}


ARCHITECTURE_LOCK_PATH = (
    RESULTS_DIR
    / "crest_final_architecture_lock.json"
)


with open(
    ARCHITECTURE_LOCK_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        ARCHITECTURE_LOCK,
        f,
        indent=2
    )


print(
    "Final architecture locked:"
)

print(
    ARCHITECTURE_LOCK_PATH
)

print(
    "\nArchitecture search: CLOSED"
)

print(
    "Fold-0 test set: UNTOUCHED"
)

Final architecture locked:
PROJECT_ROOT\results\crest_final_architecture_lock.json

Architecture search: CLOSED
Fold-0 test set: UNTOUCHED
